# 10-Armed Bandit Problem

We are implementing a **bandit problem** with **10 arms**, where:
* Each arm will have a true value **q*(a)** sampled from a Gaussian distribution with **mean 0** and **variance 1**.
* Rewards for selecting an arm **a** will be sampled from a Gaussian distribution with **mean q*(a)** and **variance 1**.

### Steps:
1. Generate **2000 independent bandit problems**.
2. Use the **sample-average method** (with incremental updates) to learn action-value estimates.
3. Simulate learning over **1000 timesteps** and analyze the performance of the following algorithms:
   * **Greedy**
   * **$\epsilon$-greedy** ($\epsilon = 0.1$ and $\epsilon = 0.01$)
4. Similarly, **simulate the UCB algorithm** on the ten-armed testbed.
5. Compare its performance with **$\epsilon$-greedy ($\epsilon = 0.1$)**.

For each algorithm, we will generate the following plots:
* **Average Reward vs. Time:** Tracks the **average reward** obtained over time.
* **% Optimal Actions vs. Time:** Tracks the **percentage of times the optimal action** is chosen over time.

## 📌 Imports

In [ ]:
# import necessary libraries here
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(63)

## 🔨 Building the Bandit Object

We start by constructing the **Bandit Object**, representing the individual arms of the bandit.

In [ ]:
# Defining bandit object
class Bandit(object):
    def __init__(self, arms=10):
        self.arms = arms
        self.q_star = np.random.normal(0, 1, self.arms)  # true action values

    def get_reward(self, action):
        return np.random.normal(self.q_star[action], 1)  # rewards sampled from N(q*(a), 1)

## Defining the Multiarm Bandit for Epsilon Greedy Approach

This implementation simulates the **multi-arm bandit problem** using the **epsilon-greedy ($\epsilon$-greedy)** method. Here's a step-by-step explanation:

### Key Parameters
1. **bandit**: The multi-arm bandit environment containing:
   * **arms**: Number of arms.
   * **q*(a)**: True reward distribution for each arm.
   * **get_reward(action)**: Method to simulate the reward for a chosen arm.
2. **timesteps**: The total number of plays or time steps.
3. **epsilon ($\epsilon$)**: The probability of choosing a random (exploratory) action.
   * With probability $1 - \epsilon$, we select the arm with the **highest estimated value** (exploitation).
   * With probability $\epsilon$, we select a **random arm** (exploration).

### Mathematical Formulation & Incremental Update Section

The estimated reward for arm $a$ is updated using the standard incremental sample-average formula:

$$Q_{k+1} = \frac{1}{k} \sum_{i=1}^{k} R_i = Q_k + \frac{1}{k} [R_k - Q_k]$$

The general form of this update rule follows:

$$\text{NewEstimate} \leftarrow \text{OldEstimate} + \text{StepSize} \left[ \text{Target} - \text{OldEstimate} \right]$$

**Action Selection:**
1. Exploitation: $action = \text{argmax}(q\_estimates)$
2. Exploration: $action = \text{randomly chosen arm}$

### Algorithm
1. **Initialization**:
   * `q_estimates`: array of zeros to store the estimated reward for each arm.
   * `action_counts`: array of zeros to track the number of times each arm is selected.
   * `rewards`: array to record rewards received at each time step.
   * `optimal_actions`: array to record whether the optimal arm was selected at each time step.
2. **Identifying the Optimal Arm**:
   * $optimal\_action = \text{argmax}(q^*(a))$
3. **Game Simulation (for t in timesteps)**:
   * Generate a random number $p \sim U(0,1)$.
   * If $p < \epsilon$: **Exploration** $\rightarrow$ choose a random arm.
   * Otherwise: **Exploitation** $\rightarrow$ choose the arm with the highest estimated reward: $action = \text{argmax}(q\_estimates)$
   * Receive reward: $reward = bandit.get\_reward(action)$
   * If the chosen action matches the optimal arm: $optimal\_actions[t] = 1$; else $0$.
   * Increment action counts and update reward estimate using the incremental formula.
4. **Return Results**:
   * `rewards`: time series of rewards at each timestep.
   * `optimal_actions`: time series indicating whether the optimal action was selected.

In [ ]:
# Defining Multiarm bandit game using epsilon greedy approach
def BanditGame(bandit, timesteps, epsilon=0):
    arms = bandit.arms
    q_estimates = np.zeros(arms)
    action_counts = np.zeros(arms)
    rewards = np.zeros(timesteps)
    optimal_actions = np.zeros(timesteps)

    optimal_action = np.argmax(bandit.q_star)

    for t in range(timesteps):
        p = np.random.rand()
        if p < epsilon:
            action = np.random.choice(arms)
        else:
            action = np.argmax(q_estimates)

        reward = bandit.get_reward(action)
        rewards[t] = reward

        if action == optimal_action:
            optimal_actions[t] = 1

        action_counts[action] += 1
        q_estimates[action] += (reward - q_estimates[action]) / action_counts[action]

    return rewards, optimal_actions

### Simulating the Game with Parameters

This section explores the simulation with different **parameters**, such as exploration strategies.

#### Insights
* **Balancing Exploration and Exploitation**:
  * **High $\epsilon$**: Encourages exploration.
  * **Low $\epsilon$**: Favors exploitation.
* **Optimal Action Tracking**:
  * Tracks how often the best action (with the highest $q^*(a)$) is selected.

This approach helps evaluate how well the $\epsilon$-greedy algorithm performs over multiple time steps and compares exploration vs. exploitation strategies.

In [ ]:
# Simulating game with different parameters
def SimulateBanditGame(n_games, arms, timesteps, epsilons):
    avg_rewards = {epsilon: np.zeros(timesteps) for epsilon in epsilons}
    avg_optimal_actions = {epsilon: np.zeros(timesteps) for epsilon in epsilons}
    i = 0
    for _ in range(n_games):
        i += 1
        print(f"Simulating game {i}/{n_games}...")
        bandit = Bandit(arms)
        for epsilon in epsilons:
            rewards, optimal_actions = BanditGame(bandit, timesteps, epsilon)
            avg_rewards[epsilon] += rewards
            avg_optimal_actions[epsilon] += optimal_actions

    for epsilon in epsilons:
        avg_rewards[epsilon] /= n_games
        avg_optimal_actions[epsilon] /= n_games

    return avg_rewards, avg_optimal_actions

### Playing Game for Greedy Approach

Run the simulation with the **Greedy algorithm**, and analyze results like average rewards and optimal actions.

In [ ]:
# Playing the Game
def PlayBanditGame(n_games, arms, timesteps, epsilons):
    avg_rewards, avg_optimal_actions = SimulateBanditGame(n_games, arms, timesteps, epsilons)
    steps = np.arange(timesteps)
    # Adjusting figure size for better clarity
    plt.figure(figsize=(10, 10))

    # Defining colors and line styles for a visually appealing distinction
    colors = {0: '#2E8B57', 0.01: '#FF4500', 0.1: '#1E90FF'}
    linestyles = {0: 'dotted', 0.01: 'dashed', 0.1: 'solid'}

    # Plotting Average Rewards
    plt.subplot(2, 1, 1)
    for epsilon, rewards in avg_rewards.items():
        label = 'Greedy' if epsilon == 0 else f'ε = {epsilon}'
        plt.plot(
            steps,
            rewards,
            label=label,
            color=colors[epsilon],
            linestyle=linestyles[epsilon],
            linewidth=2
        )
    plt.xlabel('Steps', fontsize=14, fontweight='bold', color='#333')
    plt.ylabel('Average Reward', fontsize=14, fontweight='bold', color='#333')
    plt.legend(fontsize=12)
    plt.title('Average Rewards vs. Steps', fontsize=16, fontweight='bold', color='#111')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    # Plotting % Optimal Actions
    plt.subplot(2, 1, 2)
    for epsilon, optimal_action in avg_optimal_actions.items():
        label = 'Greedy' if epsilon == 0 else f'ε = {epsilon}'
        plt.plot(
            steps,
            optimal_action * 100,
            label=label,
            color=colors[epsilon],
            linestyle=linestyles[epsilon],
            linewidth=2
        )
    plt.xlabel('Steps', fontsize=14, fontweight='bold', color='#333')
    plt.ylabel('% Optimal Actions', fontsize=14, fontweight='bold', color='#333')
    plt.legend(fontsize=12)
    plt.title('% Optimal Actions vs. Steps', fontsize=16, fontweight='bold', color='#111')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    # Tight layout and display
    plt.tight_layout(pad=3)
    plt.show()

In [ ]:
# GamePlay function call
epsilons = [0, 0.01, 0.1]
n_games = 2000
arms = 10
timesteps = 1000

PlayBanditGame(n_games, arms, timesteps, epsilons)

## Creating Multiarm Bandit for UCB

This implementation simulates the **multi-arm bandit problem** using the **Upper Confidence Bound (UCB)** algorithm. It dynamically balances **exploration** and **exploitation** by incorporating confidence intervals in action selection.

### 🔍 Key Parameters
* **bandit**: The multi-arm bandit environment containing:
  * **arms**: Number of arms.
  * **q*(a)**: True reward distribution for each arm.
  * **get_reward(action)**: Method to simulate the reward for a chosen arm.
  * **timesteps**: The total number of plays or time steps.
* **c**: A parameter controlling the degree of exploration.
  * Larger **c** values encourage exploration by widening the confidence bounds.
  * Smaller **c** values focus on exploitation.

### 🧮 Mathematical Formulation
**UCB Formula:**  
The Upper Confidence Bound is calculated as:
$$Q_{estimates}[a] + c \cdot \sqrt{\frac{\log(t + 1)}{actionCounts[a]}}$$

**Incremental Update:**  
The estimated reward for arm $a$ is updated as:
$$Q_{estimates}[a] = Q_{estimates}[a] + \frac{reward - Q_{estimates}[a]}{actionCounts[a]}$$

### 📐 Algorithm
1. **Initialization**:
   * `Q_estimates`: An array of zeros to store the estimated reward for each arm.
   * `action_counts`: An array of zeros to track the number of times each arm is selected.
   * `rewards`: An array to record rewards received at each time step.
2. **Game Simulation (for t in timesteps)**:
   * In the first **arms** steps, each arm is played once to gather initial information: $action = t$ for $t < arms$
   * Afterward, the UCB formula is used to select the arm with the highest Upper Confidence Bound: $action = \max(UCBArm)$
   * Receive reward: $reward = bandit.getReward(action)$
   * Increment the count for the chosen arm: $actionCounts[a] = actionCounts[a] + 1$
   * Update the reward estimate using the incremental formula.
3. **Return Results**:
   * `rewards`: A time series of rewards at each timestep.

### 🎯 Insights
* **Exploration vs. Exploitation**:
  * The UCB formula adds a confidence interval term to the reward estimate.
  * Arms with lower play counts are assigned wider confidence intervals, encouraging exploration.
* **Parameter c**:
  * A higher **c** value encourages more exploration by widening the confidence bounds.
  * A lower **c** value focuses on exploitation by narrowing the bounds.
* **Performance**:
  * UCB typically performs better than **$\epsilon$-greedy** as it dynamically adjusts exploration based on the number of plays and observed rewards.

In [ ]:
# Defining UCB Bandit game
def UCBBanditGame(bandit, timesteps, c=2):
    arms = bandit.arms
    Q_estimates = np.zeros(arms)
    action_counts = np.zeros(arms)
    rewards = np.zeros(timesteps)

    for t in range(timesteps):
        if t < arms:
            action = t
        else:
            action = np.argmax(Q_estimates + c * np.sqrt(np.log(t + 1) / action_counts))

        reward = bandit.get_reward(action)
        rewards[t] = reward

        action_counts[action] += 1
        Q_estimates[action] += (reward - Q_estimates[action]) / action_counts[action]

    return rewards

## Running UCB Simulations and 📊 Analyzing Output

Simulate the UCB algorithm and compare its performance with other methods like **$\epsilon$-greedy**.

In [ ]:
# play the UCB epsilon game
def PlayUCBEpsilon(n_games, arms, timesteps, c, epsilon):
    avg_rewards = {epsilon: np.zeros(timesteps)}
    avg_rewards['UCB'] = np.zeros(timesteps)
    i = 0
    for _ in range(n_games):
        i += 1
        print(f"Simulating game {i}/{n_games}...")
        bandit = Bandit(arms)
        rewards, _ = BanditGame(bandit, timesteps, epsilon)
        avg_rewards[epsilon] += rewards

        rewards_ucb = UCBBanditGame(bandit, timesteps, c)
        avg_rewards['UCB'] += rewards_ucb

    for key in avg_rewards.keys():
        avg_rewards[key] /= n_games

    steps = np.arange(timesteps)

    # Setting up the figure
    plt.figure(figsize=(10, 8))

    # Plotting Average Rewards
    plt.subplot(2, 1, 1)
    for key, rewards in avg_rewards.items():
        label = f'ε = {key}' if key != 'UCB' else 'UCB (c=2)'
        color = 'grey' if key != 'UCB' else '#1E90FF'
        linestyle = 'solid' if key == 'UCB' else 'dashed'
        plt.plot(
            steps,
            rewards,
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=2 if key == 'UCB' else 1.5
        )

    plt.xlabel('Steps', fontsize=14, fontweight='bold', color='#333')
    plt.ylabel('Average Reward', fontsize=14, fontweight='bold', color='#333')
    plt.legend(fontsize=12, loc='lower right')
    plt.title('Average Rewards vs. Steps', fontsize=16, fontweight='bold', color='#111')

    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
# GamePlay
PlayUCBEpsilon(n_games=2000, arms=10, timesteps=1000, c=2, epsilon=0.1)

## Extended Analysis: UCB vs. Epsilon-Greedy with Optimal Action Tracking & Progress Logging

In this analysis section, we update our UCB simulation pipeline to:
1. **Track Optimal Actions for UCB:** Compute and plot the percentage of optimal action selection for both UCB ($c=2$) and $\epsilon$-greedy ($\epsilon=0.1$).
2. **Monitor Simulation Progress:** Output real-time execution progress across all 2000 game runs.
3. **Export Visualization:** Save the dual-panel comparison plot locally as `ucb_epsilon_simulation.png`.

In [ ]:
# Extended UCB implementation returning both rewards and optimal actions
def UCBBanditGame(bandit, timesteps, c=2):
    arms = bandit.arms
    Q_estimates = np.zeros(arms)
    action_counts = np.zeros(arms)
    rewards = np.zeros(timesteps)
    optimal_actions = np.zeros(timesteps)

    optimal_action = np.argmax(bandit.q_star)

    for t in range(timesteps):
        if t < arms:
            action = t
        else:
            action = np.argmax(Q_estimates + c * np.sqrt(np.log(t + 1) / action_counts))

        reward = bandit.get_reward(action)
        rewards[t] = reward
        if action == optimal_action:
            optimal_actions[t] = 1
        action_counts[action] += 1
        Q_estimates[action] += (reward - Q_estimates[action]) / action_counts[action]

    return rewards, optimal_actions

In [ ]:
# Extended simulation runner with logging and plot exporting
def PlayUCBEpsilon(n_games, arms, timesteps, c, epsilon):
    avg_rewards = {epsilon: np.zeros(timesteps)}
    avg_rewards['UCB'] = np.zeros(timesteps)
    avg_optimal_actions = {epsilon: np.zeros(timesteps)}
    avg_optimal_actions['UCB'] = np.zeros(timesteps)
    i = 0
    for _ in range(n_games):
        i += 1
        print(f"Simulating game {i}/{n_games}...")
        bandit = Bandit(arms)
        rewards, optimal_actions = BanditGame(bandit, timesteps, epsilon)
        avg_rewards[epsilon] += rewards
        avg_optimal_actions[epsilon] += optimal_actions

        rewards_ucb, optimal_actions_ucb = UCBBanditGame(bandit, timesteps, c)
        avg_rewards['UCB'] += rewards_ucb
        avg_optimal_actions['UCB'] += optimal_actions_ucb

    for key in avg_rewards.keys():
        avg_rewards[key] /= n_games

    for key in avg_optimal_actions.keys():
        avg_optimal_actions[key] /= n_games

    steps = np.arange(timesteps)

    # Setting up the figure
    plt.figure(figsize=(10, 8))

    # Plotting Average Rewards
    plt.subplot(2, 1, 1)
    for key, rewards in avg_rewards.items():
        label = f'ε = {key}' if key != 'UCB' else 'UCB (c=2)'
        color = 'grey' if key != 'UCB' else '#1E90FF'
        linestyle = 'solid' if key == 'UCB' else 'dashed'
        plt.plot(
            steps,
            rewards,
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=2 if key == 'UCB' else 1.5
        )

    plt.xlabel('Steps', fontsize=14, fontweight='bold', color='#333')
    plt.ylabel('Average Reward', fontsize=14, fontweight='bold', color='#333')
    plt.legend(fontsize=12, loc='lower right')
    plt.title('Average Rewards vs. Steps', fontsize=16, fontweight='bold', color='#111')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    # Plotting % Optimal Actions
    plt.subplot(2, 1, 2)
    for key, optimal_action in avg_optimal_actions.items():
        label = f'ε = {key}' if key != 'UCB' else 'UCB (c=2)'
        color = 'grey' if key != 'UCB' else '#1E90FF'
        linestyle = 'solid' if key == 'UCB' else 'dashed'
        plt.plot(
            steps,
            optimal_action * 100,
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=2
        )
    plt.xlabel('Steps', fontsize=14, fontweight='bold', color='#333')
    plt.ylabel('% Optimal Actions', fontsize=14, fontweight='bold', color='#333')
    plt.legend(fontsize=12)
    plt.title('% Optimal Actions vs. Steps', fontsize=16, fontweight='bold', color='#111')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    plt.tight_layout(pad=3)
    plt.savefig('ucb_epsilon_simulation.png', bbox_inches='tight')
    plt.show()

In [ ]:
# Execute extended analysis simulation
PlayUCBEpsilon(n_games=2000, arms=10, timesteps=1000, c=2, epsilon=0.1)